In [0]:
from pyspark.sql.functions import col

df = spark.read.option("header", True).option("inferSchema", True).csv(
    "/Volumes/workspace/default/tseries_data/PdM_telemetry.csv"
)

df.printSchema()

root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)



In [0]:
from delta.tables import DeltaTable

table_name = "workspace.default.bronze_telemetry"

DeltaTable.createIfNotExists(spark) \
    .tableName(table_name) \
    .addColumn("datetime", "TIMESTAMP") \
    .addColumn("machineID", "INT") \
    .addColumn("volt", "DOUBLE") \
    .addColumn("rotate", "DOUBLE") \
    .addColumn("pressure", "DOUBLE") \
    .addColumn("vibration", "DOUBLE") \
    .execute()

print("Bronze telemetry Delta table created")
target = DeltaTable.forName(
    spark,
    "workspace.default.bronze_telemetry"
)

target.alias("t").merge(
    df.alias("s"),
    """
    t.machineID = s.machineID
    AND t.datetime = s.datetime
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Bronze telemetry table updated using MERGE")

Bronze telemetry Delta table created
Bronze telemetry table updated using MERGE


In [0]:
spark.table(
    "workspace.default.bronze_telemetry"
).printSchema()

print(
    "Bronze telemetry row count:",
    spark.table(
        "workspace.default.bronze_telemetry"
    ).count()
)

root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)

Bronze telemetry row count: 876100


In [0]:
df_errors = spark.read.option("header", True).option("inferSchema", True).csv(
    "/Volumes/workspace/default/tseries_data/PdM_errors.csv"
)

df_errors.printSchema()

root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- errorID: string (nullable = true)



In [0]:
from delta.tables import DeltaTable

target_table = "workspace.default.bronze_errors"

if not spark.catalog.tableExists(target_table):

    df_errors.write.format("delta").saveAsTable(target_table)

else:

    target = DeltaTable.forName(spark, target_table)

    target.alias("t").merge(
        df_errors.alias("s"),
        """
        t.machineID = s.machineID
        AND t.datetime = s.datetime
        AND t.errorID = s.errorID
        """
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

print("Bronze errors table created/updated")
df_errors.printSchema()

Bronze errors table created/updated
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- errorID: string (nullable = true)



In [0]:
df_failures = spark.read.option("header", True).option("inferSchema", True).csv(
    "/Volumes/workspace/default/tseries_data/PdM_failures.csv"
)

df_failures.printSchema()

root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- failure: string (nullable = true)



In [0]:
from delta.tables import DeltaTable

target_table = "workspace.default.bronze_failures"

if not spark.catalog.tableExists(target_table):

    df_failures.write.format("delta").saveAsTable(target_table)

else:

    target = DeltaTable.forName(spark, target_table)

    target.alias("t").merge(
        df_failures.alias("s"),
        """
        t.machineID = s.machineID
        AND t.datetime = s.datetime
        AND t.failure = s.failure
        """
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

print("Bronze failures table created/updated")
df_failures.printSchema()

Bronze failures table created/updated
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- failure: string (nullable = true)



In [0]:
df_machines = spark.read.option("header", True).option("inferSchema", True).csv(
    "/Volumes/workspace/default/tseries_data/PdM_machines.csv"
)

df_machines.printSchema()

root
 |-- machineID: integer (nullable = true)
 |-- model: string (nullable = true)
 |-- age: integer (nullable = true)



In [0]:
from delta.tables import DeltaTable

target_table = "workspace.default.bronze_machines"

if not spark.catalog.tableExists(target_table):

    df_machines.write.format("delta").saveAsTable(target_table)

else:

    target = DeltaTable.forName(spark, target_table)

    target.alias("t").merge(
        df_machines.alias("s"),
        "t.machineID = s.machineID"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

print("Bronze machines table created/updated")
df_machines.printSchema()

Bronze machines table created/updated
root
 |-- machineID: integer (nullable = true)
 |-- model: string (nullable = true)
 |-- age: integer (nullable = true)



In [0]:
df_maint = spark.read.option("header", True).option("inferSchema", True).csv(
    "/Volumes/workspace/default/tseries_data/PdM_maint.csv"
)

df_maint.printSchema()

root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- comp: string (nullable = true)



In [0]:
from delta.tables import DeltaTable

target_table = "workspace.default.bronze_maint"

if not spark.catalog.tableExists(target_table):

    df_maint.write.format("delta").saveAsTable(target_table)

else:

    target = DeltaTable.forName(spark, target_table)

    target.alias("t").merge(
        df_maint.alias("s"),
        """
        t.machineID = s.machineID
        AND t.datetime = s.datetime
        AND t.comp = s.comp
        """
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

print("Bronze maintenance table created/updated")
df_maint.printSchema()

Bronze maintenance table created/updated
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- comp: string (nullable = true)

